# Download Qwen3.5-2B

This notebook downloads `Qwen/Qwen3.5-2B` into `ABSA/model`, which is the default model folder used by `bin/zeroshot.py`.

Run it from this notebook in `ABSA/notebooks/`. Model weights are ignored by git.

## 1. Install the downloader

This only installs the Hugging Face Hub client used to download the model files. The full GPU environment for inference/training can be installed separately on Vast.

In [ ]:
%pip install -U huggingface_hub

## 2. Resolve paths

The code below finds the ABSA folder and sets `MODEL_DIR` to `ABSA/model`.

In [ ]:
from pathlib import Path

MODEL_ID = "Qwen/Qwen3.5-2B"

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, *cwd.parents]
ABSA_DIR = next(path for path in candidates if (path / "dataset" / "train.json").exists())
MODEL_DIR = ABSA_DIR / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("ABSA_DIR:", ABSA_DIR)
print("MODEL_DIR:", MODEL_DIR)

## 3. Optional login

`Qwen/Qwen3.5-2B` is public. If Hugging Face asks for authentication or you hit rate limits, uncomment and run the login cell.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## 4. Download the model

This downloads all model files into `ABSA/model`. It is several GB, so run this on the machine where you will execute inference/training.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
)

print("Downloaded", MODEL_ID, "to", MODEL_DIR)

## 5. Verify files

After this cell passes, `bin/zeroshot.py` can use the default `--model-path ../model` without extra configuration.

In [ ]:
required_files = ["config.json", "tokenizer_config.json"]
missing = [name for name in required_files if not (MODEL_DIR / name).exists()]
assert not missing, f"Missing files: {missing}"

print("Model folder looks valid.")
for path in sorted(MODEL_DIR.iterdir()):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{path.name:45s} {size_mb:8.1f} MB")

## 6. Next command

From `ABSA/`, run a tiny sanity check first:

```bash
python3 bin/zeroshot.py --limit 5 --keep-raw
python3 bin/stats.py outputs/ZS.devel.absa_v1.nothink.t1.0.p1.0.k20.mp0.0.rp1.0.m512.json
```

Then run the full development split without `--limit` once the setup is confirmed.